# 👕 Day 1 · Notebook 02 — Training a CNN on FashionMNIST

In this notebook we build a **complete, professional training pipeline** — the same skeleton you will reuse all week:

```
 Dataset ──► DataLoader ──► Model (nn.Module) ──► Loss ──► Optimizer
    ▲                                                        │
    └──────────── training loop (epochs × batches) ◄─────────┘
                        │
                        ▼
      validation ► best checkpoint (state_dict) ► test ► analysis
```

**What you will learn**
1. Load a dataset with `torchvision`, apply `transforms`, split into **train / validation / test**.
2. Batch data with `DataLoader`.
3. Define models as `nn.Module` subclasses: an **MLP baseline** and a small **CNN**.
4. Write reusable `train_one_epoch()` / `evaluate()` / `fit()` functions.
5. Plot learning curves, a **confusion matrix** and **misclassified** examples.
6. **Save** the best model's `state_dict`, **reload** it and verify it.
7. Compare MLP vs CNN: accuracy, parameter count, file size, training time.

> ⏱️ With `QUICK_RUN = False` the whole notebook takes roughly 3–5 minutes on a laptop CPU. Set `QUICK_RUN = True` for a ~30-second smoke test on a small subset.

## 0. Configuration & reproducibility

All "knobs" live in one cell at the top. That's a good habit: experiments become easy to change and to document.

**Seeds:** we seed Python, NumPy and PyTorch. This makes weight initialisation, data shuffling and dropout repeatable *on the same machine and library version*. (Bit-exact reproducibility across different hardware/GPU kernels is not guaranteed.)

In [ ]:
QUICK_RUN = False          # True -> tiny subset + 1 epoch, for a fast smoke test

import copy
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

SEED        = 42
BATCH_SIZE  = 128
LR          = 1e-3
EPOCHS_MLP  = 1 if QUICK_RUN else 3
EPOCHS_CNN  = 1 if QUICK_RUN else 5
DATA_DIR    = Path("..") / "data"        # shared data folder at the course root
MODEL_DIR   = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)      # no-op without a GPU

set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device = {device} | QUICK_RUN = {QUICK_RUN}")

## 1. The data: FashionMNIST

FashionMNIST = 70,000 greyscale **28×28** images of clothing in **10 classes** (60k train, 10k test). It is a drop-in, harder replacement for handwritten-digit MNIST.

### Transforms
A `transform` is applied to every sample when it is loaded:
* `ToTensor()` converts a PIL image (H×W, values 0–255) into a `float32` tensor of shape **(C, H, W)** with values in **[0, 1]**.
* `Normalize(mean, std)` standardises the pixels: `(x - mean) / std`. Centred inputs make optimisation easier.

First, let's compute the dataset's mean and std (from the **training** set only — never peek at the test set!).

In [ ]:
raw_train = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True)
pixels = raw_train.data.float() / 255.0          # (60000, 28, 28) uint8 -> float in [0,1]
MEAN, STD = pixels.mean().item(), pixels.std().item()
print(f"train pixel mean = {MEAN:.4f}, std = {STD:.4f}")

CLASSES = raw_train.classes
print("classes:", CLASSES)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

full_train = datasets.FashionMNIST(root=DATA_DIR, train=True,  download=True, transform=transform)
test_set   = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)

img, label = full_train[0]                      # a Dataset is indexable: returns (x, y)
print("one sample:", img.shape, img.dtype, "| label:", label, f"({CLASSES[label]})")
print(f"pixel range after normalisation: [{img.min():.2f}, {img.max():.2f}]")

### Train / validation / test split

| Split | Used for | Touch it… |
|---|---|---|
| **train** | computing gradients, updating weights | every batch |
| **validation** | choosing hyper-parameters, early stopping, picking the best epoch | after every epoch |
| **test** | the final, unbiased estimate of performance | **once**, at the very end |

If you tune on the test set, your reported accuracy becomes optimistic — it's no longer "unseen" data.
We carve 6,000 validation images out of the 60,000 training images using a **seeded** generator so the split is identical every run.

In [ ]:
n_val = 6_000
train_set, val_set = random_split(full_train, [len(full_train) - n_val, n_val],
                                  generator=torch.Generator().manual_seed(SEED))

if QUICK_RUN:   # small subsets for a fast smoke test
    train_set = Subset(train_set, range(5_000))
    val_set   = Subset(val_set,   range(1_000))
    test_set  = Subset(test_set,  range(2_000))

print(f"train: {len(train_set):,} | val: {len(val_set):,} | test: {len(test_set):,}")

### DataLoader

A `DataLoader` wraps a `Dataset` and yields **mini-batches**:
* `batch_size` — how many samples per gradient step,
* `shuffle=True` for training (so batches differ every epoch — this helps SGD); **no** shuffling for val/test (not needed, and keeps results deterministic),
* `num_workers` — parallel loading processes. We use `0` here: on Windows/macOS inside Jupyter, worker processes can cause issues, and FashionMNIST is small enough anyway. In scripts on Linux, `num_workers=2–8` speeds up loading.

In [ ]:
g = torch.Generator().manual_seed(SEED)       # makes shuffling reproducible
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=g, num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=512, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=512, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print("batch of images:", xb.shape, xb.dtype)   # (B, C, H, W)
print("batch of labels:", yb.shape, yb.dtype)   # (B,)  int64 -> what CrossEntropyLoss wants
print("batches per epoch:", len(train_loader))

### Look at the data!
Always visualise a few samples before training — it catches wrong labels, wrong normalisation, wrong channel order, etc. We *un-normalise* (`x * std + mean`) for display.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, img, lbl in zip(axes.flat, xb[:16], yb[:16]):
    ax.imshow((img.squeeze(0) * STD + MEAN).numpy(), cmap="gray")
    ax.set_title(CLASSES[lbl], fontsize=8)
    ax.axis("off")
plt.suptitle("Random training samples")
plt.tight_layout(); plt.show()

## 2. Models as `nn.Module`

Every PyTorch model is a subclass of `nn.Module` with two responsibilities:
1. `__init__`: create the layers (their parameters are registered automatically),
2. `forward(x)`: describe how input flows through the layers. **Never call `forward` directly** — call `model(x)`, which also runs hooks.

### 2.1 Baseline: a Multi-Layer Perceptron (MLP)
An MLP flattens the image into a 784-vector and ignores the 2-D structure. It's a sensible baseline.

```
(B,1,28,28) ─Flatten─► (B,784) ─Linear─► 256 ─ReLU─► ─Linear─► 128 ─ReLU─► ─Linear─► (B,10) logits
```

The final layer outputs **logits** (unnormalised scores). We do **not** add a softmax: `nn.CrossEntropyLoss` applies `log_softmax` internally (numerically more stable).

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(),
            nn.Linear(256, 128),     nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(MLP())
print(f"trainable parameters: {count_params(MLP()):,}")

### 2.2 A small Convolutional Neural Network (CNN)

Convolutions slide small learnable filters (here 3×3) over the image. They exploit two facts about images:
* **locality** — nearby pixels are related,
* **translation equivariance** — a sleeve is a sleeve wherever it appears.

Weight sharing makes convolutions very parameter-efficient. Max-pooling halves the spatial size, giving later layers a wider "view".

```
input      conv3x3(32)+ReLU   maxpool   conv3x3(64)+ReLU   maxpool   flatten   FC 128   FC 10
1×28×28 ─► 32×28×28 ────────► 32×14×14 ─► 64×14×14 ──────► 64×7×7 ─► 3136 ───► 128 ──► 10
                   └────────── features ───────────────────────────┘ └──── classifier ───┘
```
`padding=1` with a 3×3 kernel keeps height/width unchanged. `Dropout` randomly zeroes activations during training to reduce overfitting.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                    # 28 -> 14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                    # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

print(f"trainable parameters: {count_params(SmallCNN()):,}")

**Shape walk-through.** A dummy batch through each layer — the best way to debug `mat1 and mat2 shapes cannot be multiplied` errors.

In [ ]:
x = torch.zeros(2, 1, 28, 28)
cnn_demo = SmallCNN()
print(f"{'input':>12s}: {tuple(x.shape)}")
for layer in list(cnn_demo.features) + list(cnn_demo.classifier):
    x = layer(x)
    print(f"{layer.__class__.__name__:>12s}: {tuple(x.shape)}")

## 3. Loss, training and evaluation functions

**Loss:** `nn.CrossEntropyLoss` for multi-class classification. It expects
* input: logits of shape `(B, n_classes)`, `float32`
* target: class indices of shape `(B,)`, `int64`

**Two modes of a model:**
* `model.train()` — dropout active, BatchNorm updates running statistics,
* `model.eval()` — dropout off, BatchNorm uses stored statistics.

Forgetting `model.eval()` at evaluation time is one of the most common bugs! And during evaluation we also use `torch.no_grad()` to save memory/time.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()              # 1. reset gradients
        logits = model(xb)                 # 2. forward
        loss = loss_fn(logits, yb)         # 3. loss (mean over the batch)
        loss.backward()                    # 4. backward
        optimizer.step()                   # 5. update
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += xb.size(0)
    return total_loss / n, correct / n


@torch.no_grad()                           # decorator form of `with torch.no_grad():`
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += xb.size(0)
    return total_loss / n, correct / n

`fit()` runs several epochs, records a **history** for plotting, and keeps a copy of the weights from the epoch with the **best validation accuracy** (a simple form of *checkpointing*). We use `copy.deepcopy` because `state_dict()` returns references to the live tensors, which would keep changing as training continues.

In [ ]:
def fit(model, train_loader, val_loader, epochs, lr=LR, device=device):
    model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_state = 0.0, None
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        te = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
        for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[k].append(v)
        flag = ""
        if va_acc > best_acc:
            best_acc, best_state, flag = va_acc, copy.deepcopy(model.state_dict()), " ⭐ best"
        print(f"epoch {epoch:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f} | {time.time() - te:5.1f}s{flag}")
    history["train_time"] = time.time() - t0
    history["best_val_acc"] = best_acc
    return history, best_state

## 4. Train the MLP baseline

In [ ]:
set_seed()
mlp = MLP()
hist_mlp, best_mlp_state = fit(mlp, train_loader, val_loader, epochs=EPOCHS_MLP)
print(f"\nMLP: best val acc {hist_mlp['best_val_acc']:.4f} in {hist_mlp['train_time']:.1f}s")

## 5. Train the CNN
Expect each epoch to take noticeably longer than the MLP (convolutions are more compute-heavy), but the accuracy should be clearly better (~90%+ after 5 epochs).

In [ ]:
set_seed()
cnn = SmallCNN()
hist_cnn, best_cnn_state = fit(cnn, train_loader, val_loader, epochs=EPOCHS_CNN)
print(f"\nCNN: best val acc {hist_cnn['best_val_acc']:.4f} in {hist_cnn['train_time']:.1f}s")

## 6. Learning curves

How to read them:
* **Both losses decreasing** → still learning; more epochs may help.
* **Train loss ↓ but val loss ↑** (gap widening) → **overfitting**: the model memorises the training set.
* **Both high and flat** → **underfitting**: model too small, LR wrong, or a bug.

Note: train metrics are averaged *during* the epoch with dropout on, while val metrics are computed *after* the epoch with dropout off — so val can even be slightly better than train early on.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, h, c in [("MLP", hist_mlp, "tab:blue"), ("CNN", hist_cnn, "tab:orange")]:
    ep = range(1, len(h["train_loss"]) + 1)
    ax[0].plot(ep, h["train_loss"], "--", color=c, label=f"{name} train")
    ax[0].plot(ep, h["val_loss"],   "-",  color=c, label=f"{name} val")
    ax[1].plot(ep, h["train_acc"],  "--", color=c, label=f"{name} train")
    ax[1].plot(ep, h["val_acc"],    "-",  color=c, label=f"{name} val")
ax[0].set_title("Loss"); ax[1].set_title("Accuracy")
for a in ax:
    a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
    a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout(); plt.show()

## 7. Save the best CNN — and reload it

**Best practice:** save the `state_dict` (an ordered dict *parameter name → tensor*), not the whole pickled model object.
* It's portable and robust to code refactoring.
* To load, you need the **class definition** + `load_state_dict()`.
* `torch.load(..., weights_only=True)` (default since PyTorch 2.6) only unpickles tensors — safer for files from untrusted sources.

`map_location` lets you load a GPU-trained checkpoint on a CPU-only machine.

In [ ]:
MODEL_PATH = MODEL_DIR / "fashion_cnn.pth"
torch.save(best_cnn_state, MODEL_PATH)
print(f"saved -> {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1e6:.2f} MB)")
print("keys in state_dict:", list(best_cnn_state.keys()))

In [ ]:
reloaded = SmallCNN()                                                   # 1. fresh instance of the class
reloaded.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))  # 2. load weights
reloaded.to(device).eval()                                              # 3. eval mode!

cnn.load_state_dict(best_cnn_state)       # put the best weights back into the in-memory model too
cnn.eval()
with torch.no_grad():
    xb, _ = next(iter(test_loader))
    xb = xb.to(device)
    same = torch.allclose(cnn(xb), reloaded(xb))
print("reloaded model gives identical outputs:", same)

loss_fn = nn.CrossEntropyLoss()
test_loss, test_acc_cnn = evaluate(reloaded, test_loader, loss_fn, device)
print(f"CNN  TEST accuracy: {test_acc_cnn:.4f}  (loss {test_loss:.4f})")

mlp.load_state_dict(best_mlp_state)
_, test_acc_mlp = evaluate(mlp, test_loader, loss_fn, device)
print(f"MLP  TEST accuracy: {test_acc_mlp:.4f}")

## 8. Error analysis

Accuracy is one number; it hides *which* classes get confused. First, collect all test predictions.

In [ ]:
@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    preds, labels = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(device)).argmax(dim=1).cpu())
        labels.append(yb)
    return torch.cat(preds), torch.cat(labels)

y_pred, y_true = predict_all(reloaded, test_loader, device)
print("predictions:", y_pred.shape, "| accuracy check:", (y_pred == y_true).float().mean().item())

### Confusion matrix
Row = true class, column = predicted class. The diagonal holds correct predictions. Off-diagonal cells show systematic confusions.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true.numpy(), y_pred.numpy(), labels=list(range(10)))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("CNN confusion matrix (test set)")
plt.tight_layout(); plt.show()

per_class = cm.diagonal() / cm.sum(axis=1)
for cls, acc in sorted(zip(CLASSES, per_class), key=lambda t: t[1]):
    print(f"{cls:12s} {acc:.3f}")

**Discussion:** Which classes are hardest? (Typically *Shirt* vs *T-shirt/top*, *Pullover* and *Coat* — even humans struggle at 28×28 pixels.) What could help: more epochs, a bigger model, data augmentation, or a **pretrained** network (→ Day 2)?

### Misclassified examples
Looking at errors often reveals label noise or genuinely ambiguous images.

In [ ]:
wrong = torch.nonzero(y_pred != y_true).flatten()
print(f"{len(wrong)} misclassified out of {len(y_true)} test images")

base_test = test_set.dataset if isinstance(test_set, Subset) else test_set
idx_map = (lambda i: test_set.indices[i]) if isinstance(test_set, Subset) else (lambda i: i)

fig, axes = plt.subplots(2, 8, figsize=(13, 4.8))
for ax, i in zip(axes.flat, wrong[:16].tolist()):
    img, _ = base_test[idx_map(i)]
    ax.imshow((img.squeeze(0) * STD + MEAN).numpy(), cmap="gray")
    ax.set_title(f"T: {CLASSES[y_true[i]]}\nP: {CLASSES[y_pred[i]]}", fontsize=7, color="red")
    ax.axis("off")
plt.suptitle("Misclassified test images (T = true, P = predicted)")
plt.tight_layout(); plt.show()

## 9. MLP vs CNN — summary table

In [ ]:
torch.save(best_mlp_state, MODEL_DIR / "fashion_mlp.pth")
summary = pd.DataFrame([
    {"model": "MLP", "params": count_params(mlp), "epochs": EPOCHS_MLP,
     "best val acc": hist_mlp["best_val_acc"], "test acc": test_acc_mlp,
     "train time (s)": hist_mlp["train_time"],
     "file size (MB)": (MODEL_DIR / "fashion_mlp.pth").stat().st_size / 1e6},
    {"model": "SmallCNN", "params": count_params(cnn), "epochs": EPOCHS_CNN,
     "best val acc": hist_cnn["best_val_acc"], "test acc": test_acc_cnn,
     "train time (s)": hist_cnn["train_time"],
     "file size (MB)": MODEL_PATH.stat().st_size / 1e6},
])
summary.style.format({"params": "{:,}", "best val acc": "{:.4f}", "test acc": "{:.4f}",
                      "train time (s)": "{:.1f}", "file size (MB)": "{:.2f}"}).hide(axis="index")

**Observations to discuss**
* The CNN is clearly more accurate. Is that only because it has more parameters (≈420k vs ≈235k)? No: ~95% of the CNN's parameters sit in its *fully-connected* layer, and the two conv layers have fewer than 19k weights. A wider MLP with the same parameter budget still does not reach CNN accuracy. The **inductive bias** of convolutions (locality + weight sharing) is what pays off.
* The CNN is slower per epoch on a CPU: FLOPs ≠ parameters. Convolutions reuse each weight at every spatial position.
* Model **file size** is ≈ `params × 4 bytes` (float32). On **Day 2** we'll shrink this ~4× with **int8 quantization** and measure the speed/accuracy trade-off.

---
## ✅ Key takeaways
* Pipeline: `Dataset` → `DataLoader` → `nn.Module` → loss → optimizer → loop.
* Separate **train / val / test**; choose the best epoch on **val**, report on **test** once.
* `model.train()` / `model.eval()` + `torch.no_grad()` for evaluation.
* Save/load **`state_dict`**; recreate the class, then `load_state_dict`.
* Visualise data, curves, confusion matrix and errors — not just a single accuracy number.

➡️ Next: **Notebook 03 — Student lab**: optimizers, regularisation, early stopping, augmentation, custom datasets and LR schedulers.